# Formal Language & Automata (21CSC301T)
## Project: Automata-Based Email Spam Detection and Classification Using Finite Automata and Regular Expressions
**B.Tech Information Technology - Third Year Academic Coursework**
---
This Google Colab notebook implements the complete research pipeline:
1. Dataset loading and normalization (Kaggle: `ssssws/spam-email-detection-dataset-clean-and-ml-ready`)
2. 10 Formal Automata & Regular Expression pattern experiments
3. Rule-based weighted scoring engine with integrated automaton simulation
4. Optional Machine Learning baseline (TF-IDF + Logistic Regression)
5. Comparative evaluation (Confusion Matrices, Precision, Recall, F1)

In [ ]:
# Step 1: Install and Import Dependencies
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
print('All libraries imported successfully!')

### Step 2: Load Sample / Kaggle Dataset
If running on Colab, upload `dataset.csv` or load the built-in sample data.

In [ ]:
# Dataset Loader with fallback sample
sample_data = [
    {'text': 'Congratulations! You won a free prize. Click here to claim your cash bonus of $5000 now!', 'label': 1},
    {'text': 'URGENT SECURITY ALERT: Verify your account password immediately at http://192.168.1.1/login', 'label': 1},
    {'text': 'Claim $1000 dollars free cash reward today! Click below to collect jackpot.', 'label': 1},
    {'text': 'Act now! Free complimentary entry into our million dollar cash prize draw.', 'label': 1},
    {'text': 'The department project review meeting is scheduled for tomorrow at 10 AM.', 'label': 0},
    {'text': 'Please find attached the lab assignment report for Formal Languages and Automata.', 'label': 0},
    {'text': 'Can you share the lecture slides from yesterday\'s compiler design seminar?', 'label': 0},
    {'text': 'Your library book loan period expires on Monday. Please renew online if needed.', 'label': 0}
]

import os
if os.path.exists('dataset.csv'):
    df = pd.read_csv('dataset.csv')
    text_col = 'email_text' if 'email_text' in df.columns else 'text'
    df = df.rename(columns={text_col: 'text'})
    print(f'Loaded Kaggle dataset with {len(df)} records.')
else:
    df = pd.DataFrame(sample_data)
    print('Using built-in sample dataset for demo.')

print('Spam count:', (df['label'] == 1).sum(), 'Ham count:', (df['label'] == 0).sum())

### Step 3: Define the 10 Formal Automata & Regex Patterns

In [ ]:
PATTERNS = {
    'EXP01_FREE': (re.compile(r'\b(free|freebie|complimentary)\b', re.IGNORECASE), 20),
    'EXP02_WIN_PRIZE': (re.compile(r'\b(win|winner|won|winning|prize|reward|jackpot)\b', re.IGNORECASE), 25),
    'EXP03_URGENT': (re.compile(r'\b(urgent|urgently|act now|expires? today|immediate action)\b', re.IGNORECASE), 20),
    'EXP04_CLICK_HERE': (re.compile(r'\b(click here|click link|click below|tap here)\b', re.IGNORECASE), 20),
    'EXP05_SUSPICIOUS_URL': (re.compile(r'(https?://\S+|www\.\S+|\b\d{1,3}(\.\d{1,3}){3}\b)', re.IGNORECASE), 25),
    'EXP06_MONEY_REWARD': (re.compile(r'(\$\s*\d+|\b\d+\s*(dollars?|usd|cash|bonus)\b)', re.IGNORECASE), 20),
    'EXP07_ACCOUNT_VERIFY': (re.compile(r'\b(verify (your )?account|security alert|reset (your )?password|account suspension)\b', re.IGNORECASE), 30),
    'EXP08_EXCLAMATIONS': (re.compile(r'!{2,}'), 15)
}
THRESHOLD = 40
print('Configured 10 automata pattern rules with threshold =', THRESHOLD)

### Step 4: Rule-Based Automata Classification Engine

In [ ]:
def classify_email_automata(text, threshold=THRESHOLD):
    text_clean = re.sub(r'\s+', ' ', str(text)).strip()
    if not text_clean:
        return {'is_spam': False, 'verdict': 'HAM', 'score': 0, 'rules': []}
    
    score = 0
    triggered = []
    for key, (pattern, weight) in PATTERNS.items():
        if pattern.search(text_clean):
            score += weight
            triggered.append(key)
    
    # Multi-pattern bonus
    if len(triggered) >= 2:
        score += 15
        triggered.append('EXP09_MULTI_PATTERN')
        
    is_spam = score >= threshold
    return {
        'is_spam': is_spam,
        'verdict': 'SPAM' if is_spam else 'HAM',
        'score': score,
        'rules': triggered
    }

# Test sample
test_msg = 'Congratulations! You won a free prize. Click here now to claim $5000!!'
print('Analysis of test email:')
print(classify_email_automata(test_msg))

### Step 5: Comparative Evaluation: Rule-Based vs Machine Learning

In [ ]:
# Run evaluation
rule_preds = [1 if classify_email_automata(t)['is_spam'] else 0 for t in df['text']]
cm = confusion_matrix(df['label'], rule_preds)
acc = accuracy_score(df['label'], rule_preds)
prec = precision_score(df['label'], rule_preds, zero_division=0)
rec = recall_score(df['label'], rule_preds, zero_division=0)
f1 = f1_score(df['label'], rule_preds, zero_division=0)

print('=== RULE-BASED AUTOMATA EVALUATION RESULTS ===')
print(f'Accuracy : {acc * 100:.2f}%')
print(f'Precision: {prec * 100:.2f}%')
print(f'Recall   : {rec * 100:.2f}%')
print(f'F1-Score : {f1 * 100:.2f}%')
print('Confusion Matrix:\n', cm)